# PREDIÇÃO DO BLOCO DE MARLIM
Para cada modelo em `../Model/Backup/`, prediz os patches `../Dataset/marlim/patches_*`
(os `.dat` já estão em [0,1] — **nenhum pré-processamento extra**) e salva as
probabilidades (sigmoid, float32) em `../Model/Backup/<modelo>/marlim/patches_<id>/masks/*.dat`.
Depois, avalie no `Analysis.ipynb` (adicione a pasta no json `PREDICTIONS`).

In [ ]:
import os, sys, glob, json, gc
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

sys.path.append('../Model')                 # Network/, utils/ vivem em Model/
from Network.index import ModelNetwork

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # nome da GPU

In [ ]:
BACKUP_DIR = '../Model/Backup'
MARLIM_DIR = '../Dataset/marlim'
PATCH_IDS  = ['1200', '1300', '1400', '2600']

models = sorted(os.listdir(BACKUP_DIR))
models

In [ ]:
def getFiles(path, limit=None, shuffle=False):
    target = sorted([os.path.abspath(p) for p in glob.glob(os.path.join(path, '*'))])
    if shuffle:
        np.random.shuffle(target)
    return target[:limit]


def getDAT(path):
    return np.fromfile(path, dtype=np.float32).reshape((128, 128, 128))


class PredictDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        img = getDAT(row.img_path)
        return torch.tensor(img, dtype=torch.float32).unsqueeze(0)

In [ ]:
class MarlimPredictor:
    """Prediz os patches de Marlim com cada modelo salvo em Backup/."""

    def __init__(self, backupDir=BACKUP_DIR, marlimDir=MARLIM_DIR, patchIds=PATCH_IDS):
        self.backupDir = backupDir
        self.marlimDir = marlimDir
        self.patchIds  = patchIds

    def loadNetwork(self, modelName):
        modelPath = f'{self.backupDir}/{modelName}'

        with open(f'{modelPath}/info.json', 'r', encoding='utf-8') as file:
            modelInfo = json.load(file)

        modelOptions = modelInfo.get('model', {})
        print(f'[{modelName}] Model Options:')
        print(json.dumps(modelOptions, indent=4))

        network   = ModelNetwork(**modelOptions)
        modelData = torch.load(f'{modelPath}/model.pth')

        network.model.load_state_dict(modelData['model'])
        network.model.eval()
        network.model.to(network.device)
        print(f'Weights from {modelPath} loaded successfully!\n')
        return network

    def getLoader(self, pid):
        imgPaths = [p for p in getFiles(f'{self.marlimDir}/patches_{pid}') if p.endswith('.dat')]
        df = pd.DataFrame({'img_path': imgPaths})

        loader = DataLoader(
            PredictDataset(df),
            batch_size=1,
            shuffle=False,
            num_workers=2,
            pin_memory=True if torch.cuda.is_available() else False
        )
        return df, loader

    def predictPatch(self, network, modelName, pid):
        df, loader = self.getLoader(pid)
        outputDir  = os.path.join(self.backupDir, modelName, 'marlim', f'patches_{pid}', 'masks')
        os.makedirs(outputDir, exist_ok=True)

        with torch.no_grad():
            for index, imgTensor in enumerate(tqdm(loader, desc=f'{modelName} - patches_{pid}')):
                logits = network.model(imgTensor.to(network.device))
                probs  = torch.softmax(logits, dim=1) if network.multiclass else torch.sigmoid(logits)
                probs  = probs.squeeze().cpu().numpy().astype(np.float32)

                filename = os.path.basename(df.iloc[index].img_path)
                probs.tofile(os.path.join(outputDir, filename))

        print(f'Predições de {modelName} - patches_{pid} salvas em {outputDir}')

    def start(self):
        for modelName in models:
            if not os.path.exists(f'{self.backupDir}/{modelName}/model.pth'):
                print(f'[{modelName}] sem model.pth — pulando')
                continue

            network = self.loadNetwork(modelName)
            for pid in self.patchIds:
                self.predictPatch(network, modelName, pid)

            del network
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


predictor = MarlimPredictor()
predictor.start()